# Monitoreo del modelo Adult Income

Este notebook ejecuta el flujo completo de monitoreo en orden con Python 3.13.14. Ejecute las celdas de arriba hacia abajo.

> Antes de abrir Jupyter, active el entorno virtual desde PowerShell con `./.venv/Scripts/Activate.ps1` y abra el notebook usando ese entorno.

## Mapa de la rúbrica y resultados esperados

| Apartado | Qué demuestra | Resultado principal |
|---|---|---|
| **O1. System Monitoring** | Latency, throughput, error rate y availability | Respuesta de `/monitoring/system` |
| **O2. Data Monitoring** | Comparación entre referencia y producción | PSI, KS, Wasserstein y categorías desconocidas |
| **O3. Model Monitoring** | Evolución del clasificador por lote | Precision, recall, F1 y ROC-AUC |
| **P. Producción y drift** | Cambio progresivo en P(X) | Batches 1–6 y niveles OK/WARNING/ALERT |
| **Q. Calidad** | Contaminación controlada de una copia | Detecta → Bloquea → Registra los seis defectos |
| **R. Reentrenamiento** | Decisión condicionada, no automática | PSI + F1 + volumen + aprobación manual |

> Aunque algunas métricas se presentan juntas por lote, cada apartado tiene una salida separada debajo.

## 1. Verificar el entorno y la carpeta del proyecto

La ruta mostrada debe terminar en `respositorio25-08-2026` y el ejecutable debe pertenecer a `.venv`.

In [32]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
print('Carpeta:', PROJECT_DIR)
print('Python:', sys.executable)
print('Versión:', sys.version.split()[0])

required = [
    PROJECT_DIR / 'requirements-monitoring.txt',
    PROJECT_DIR / 'src' / 'monitoring.py',
    PROJECT_DIR / 'src' / 'simulate_production.py',
    PROJECT_DIR / 'resultado_pipeline' / 'adult_clean.csv',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Faltan archivos: {missing}'
assert '.venv' in sys.executable, 'Seleccione como kernel el Python de .venv.'
print('Entorno y archivos verificados correctamente.')

Carpeta: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
Python: c:\Users\c3283\Desktop\INCOEX\fase6 proyecto integrador\.venv\Scripts\python.exe
Versión: 3.13.14
Entorno y archivos verificados correctamente.


## 2. Instalar dependencias

`%pip` instala las dependencias en el mismo entorno utilizado por el notebook.

In [33]:
%pip install -r requirements-monitoring.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Ejecutar las pruebas de monitoreo

El resultado esperado es `12 passed`.

In [34]:
!"{sys.executable}" -m pytest tests/test_monitoring.py -v -p no:cacheprovider

============================= test session starts =============================
platform win32 -- Python 3.13.14, pytest-8.3.3, pluggy-1.6.0 -- c:\Users\c3283\Desktop\INCOEX\fase6 proyecto integrador\.venv\Scripts\python.exe
rootdir: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
plugins: anyio-4.14.2
collecting ... collected 12 items

tests/test_monitoring.py::test_numeric_psi_is_zero_for_equal_distributions PASSED [  8%]
tests/test_monitoring.py::test_numeric_psi_detects_high_drift PASSED     [ 16%]
tests/test_monitoring.py::test_categorical_psi_is_zero_for_equal_distributions PASSED [ 25%]
tests/test_monitoring.py::test_categorical_psi_detects_high_drift PASSED [ 33%]
tests/test_monitoring.py::test_data_monitoring_detects_numeric_drift PASSED [ 41%]
tests/test_monitoring.py::test_data_monitoring_reports_missing_columns PASSED [ 50%]
tests/test_monitoring.py::test_model_monitoring_calculates_classification_metrics PASSED [ 58%]
tests/test_monitoring.py::test_model_monitoring_wi

## 4. Ejecutar todas las pruebas del proyecto

Esta comprobación es más amplia. El resultado esperado actualmente es `77 passed`.

In [35]:
!"{sys.executable}" -m pytest tests -v -p no:cacheprovider

============================= test session starts =============================
platform win32 -- Python 3.13.14, pytest-8.3.3, pluggy-1.6.0 -- c:\Users\c3283\Desktop\INCOEX\fase6 proyecto integrador\.venv\Scripts\python.exe
rootdir: c:\Users\c3283\Desktop\INCOEX\respositorio25-08-2026
plugins: anyio-4.14.2
collecting ... collected 84 items

tests/test_api.py::test_health_returns_200 PASSED                        [  1%]
tests/test_api.py::test_health_returns_model_metadata PASSED             [  2%]
tests/test_api.py::test_valid_request_returns_200 PASSED                 [  3%]
tests/test_api.py::test_valid_request_returns_expected_schema PASSED     [  4%]
tests/test_api.py::test_prediction_is_binary PASSED                      [  5%]
tests/test_api.py::test_probability_is_valid PASSED                      [  7%]
tests/test_api.py::test_model_version_is_valid PASSED                    [  8%]
tests/test_api.py::test_out_of_range_values_return_422[age-16] PASSED    [  9%]
tests/test_api.p

## O2. Data Monitoring

Compara $P_{reference}(X)$ contra $P_{production}(X)$ mediante PSI, Kolmogorov-Smirnov y distancia Wasserstein.

## P. Simulación de producción y drift

Se generan 6 lotes de 1.000 filas con semilla 42. El drift aumenta progresivamente entre los lotes. Los niveles usados son PSI < 0.10 = **OK**, 0.10 ≤ PSI < 0.25 = **WARNING** y PSI ≥ 0.25 = **ALERT**. Son umbrales configurables y deben recalibrarse si cambian el modelo, los datos o el riesgo aceptado.

In [36]:
!"{sys.executable}" -m src.simulate_production --batches 6 --batch-size 1000 --random-state 42

 batch_id  drift_strength  drift_detected             high_drift_features  precision   recall       f1  roc_auc
        1             0.0           False                                   0.744444 0.848101 0.792899 0.960430
        2             0.2           False                                   0.560117 0.760956 0.645270 0.850068
        3             0.4            True     capital-gain,hours-per-week   0.477833 0.772908 0.590563 0.796754
        4             0.6            True     capital-gain,hours-per-week   0.407563 0.757812 0.530055 0.734443
        5             0.8            True age,capital-gain,hours-per-week   0.302326 0.793427 0.437824 0.695188
        6             1.0            True age,capital-gain,hours-per-week   0.287902 0.764228 0.418242 0.642228


## O3. Model Monitoring

Para cada batch se reportan precision, recall, F1 y ROC-AUC. La misma tabla conserva las columnas de O2 para contrastar drift y desempeño, pero son dimensiones diferentes.

## 6. Revisar el resumen de los lotes

La tabla permite observar desde qué lote aparece drift y cómo cambian las métricas del modelo.

In [37]:
import pandas as pd

summary_path = PROJECT_DIR / 'resultado_pipeline' / 'monitoring' / 'monitoring_summary.csv'
summary = pd.read_csv(summary_path)
print('O2/P — Resultados de drift por batch:')
display(summary[['batch_id', 'drift_strength', 'drift_detected', 'high_drift_features']])
print('O3 — Resultados del modelo por batch:')
display(summary[['batch_id', 'precision', 'recall', 'f1', 'roc_auc']])

O2/P — Resultados de drift por batch:


,batch_id,drift_strength,drift_detected,high_drift_features
0,1,0.0,False,NaN
1,2,0.2,False,NaN
2,3,0.4,True,"capital-gain,hours-per-week"
3,4,0.6,True,"capital-gain,hours-per-week"
4,5,0.8,True,"age,capital-gain,hours-per-week"
5,6,1.0,True,"age,capital-gain,hours-per-week"


O3 — Resultados del modelo por batch:


,batch_id,precision,recall,f1,roc_auc
0,1,0.744444,0.848101,0.792899,0.960430
1,2,0.560117,0.760956,0.645270,0.850068
2,3,0.477833,0.772908,0.590563,0.796754
3,4,0.407563,0.757812,0.530055,0.734443
4,5,0.302326,0.793427,0.437824,0.695188
5,6,0.287902,0.764228,0.418242,0.642228


## 7. Generar el reporte general

El reporte compara el dataset de referencia con los 6.000 registros simulados y guarda el resultado en JSON.

In [38]:
!"{sys.executable}" -m src.monitoring --reference resultado_pipeline/adult_clean.csv --production resultado_pipeline/monitoring/production_batch.csv --output resultado_pipeline/monitoring/monitoring_report.json

{
  "reference_path": "resultado_pipeline\\adult_clean.csv",
  "production_path": "resultado_pipeline\\monitoring\\production_batch.csv",
  "data_monitoring": {
    "reference_rows": 48790,
    "production_rows": 6000,
    "numeric_features": {
      "age": {
        "status": "ok",
        "reference_count": 48790,
        "production_count": 6000,
        "reference_mean": 38.652797704447636,
        "production_mean": 44.53033333333333,
        "reference_missing_rate": 0.0,
        "production_missing_rate": 0.0,
        "psi": 0.15991157509400258,
        "psi_level": "drift_moderado",
        "ks_statistic": 0.1548081232492997,
        "ks_pvalue": 2.7509725806313195e-112,
        "ks_drift_detected": true,
        "wasserstein_distance": 5.877535628885701
      },
      "fnlwgt": {
        "status": "ok",
        "reference_count": 48790,
        "production_count": 6000,
        "reference_mean": 189668.9993646239,
        "production_mean": 187620.537,
        "reference_missi

## 8. Interpretar el reporte

Se muestran las variables con drift alto y las métricas generales del modelo.

In [39]:
import json

report_path = PROJECT_DIR / 'resultado_pipeline' / 'monitoring' / 'monitoring_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))

print('Drift detectado:', report['data_monitoring']['drift_detected'])
print('Variables con drift alto:', report['data_monitoring']['high_drift_features'])
print('Métricas del modelo:')
display(pd.Series(report['model_monitoring']))

Drift detectado: True
Variables con drift alto: ['capital-gain', 'hours-per-week']
Métricas del modelo:


status                      evaluated
rows                             6000
valid_predictions                6000
positive_prediction_rate     0.450833
average_probability          0.499219
probability_std              0.424364
ground_truth_available           True
evaluated_rows                   6000
precision                    0.420333
recall                       0.781981
f1                           0.546766
roc_auc                      0.760065
dtype: object

## O1. System Monitoring

La API registra latency, throughput, error rate y availability. Primero se inicia la API, después se realizan consultas y finalmente se visualiza `/monitoring/system`.

## 9. Iniciar la API en el puerto 8001

La siguiente celda inicia Uvicorn en segundo plano, sin `--reload`, para evitar procesos adicionales dentro del notebook. Si el puerto ya está ocupado, primero detenga la otra API.

In [50]:
import subprocess
import time
import urllib.request

if 'api_process' in globals() and api_process.poll() is None:
    print('La API iniciada por este notebook ya está activa.')
else:
    api_process = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'src.api.main:app', '--host', '127.0.0.1', '--port', '8001'],
        cwd=PROJECT_DIR,
    )
    for _ in range(20):
        if api_process.poll() is not None:
            raise RuntimeError('La API no pudo iniciar. Revise si el puerto 8001 está ocupado.')
        try:
            with urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=1) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(0.5)
    else:
        raise RuntimeError('La API no respondió dentro del tiempo esperado.')
    print('API activa en http://127.0.0.1:8001')

La API iniciada por este notebook ya está activa.


## 10. Consultar salud y métricas del sistema

Rutas disponibles:

- [Estado del modelo](http://127.0.0.1:8001/health)
- [Métricas del sistema](http://127.0.0.1:8001/monitoring/system)
- [Swagger](http://127.0.0.1:8001/docs)
- [Especificación OpenAPI](http://127.0.0.1:8001/openapi.json)

In [42]:
def get_json(url):
    with urllib.request.urlopen(url, timeout=5) as response:
        return json.loads(response.read().decode('utf-8'))

print('Estado del modelo:')
display(get_json('http://127.0.0.1:8001/health'))
print('Métricas del sistema:')
display(get_json('http://127.0.0.1:8001/monitoring/system'))

Estado del modelo:


{'status': 'ok',
 'algorithm': 'hist_gradient_boosting',
 'threshold': 0.645,
 'model_version': '225a40e0'}

Métricas del sistema:


{'total_requests': 1,
 'successful_requests': 1,
 'error_requests': 0,
 'uptime_seconds': 11.6331,
 'availability': 1.0,
 'error_rate': 0.0,
 'throughput_requests_per_second': 0.085962,
 'average_latency_ms': 646.9065,
 'p95_latency_ms': 646.9065}

## 11. Detener la API iniciada por el notebook

Ejecute esta celda cuando termine. Solo detiene el proceso creado en la sección 9.

In [43]:
if 'api_process' in globals() and api_process.poll() is None:
    api_process.terminate()
    api_process.wait(timeout=10)
    print('API detenida.')
else:
    print('No hay una API iniciada por este notebook.')

API detenida.


## Q. Simulación obligatoria de problemas de calidad

Esta prueba trabaja sobre una copia en memoria del batch 1 e introduce missing values, una fila duplicada, un outlier extremo, un datatype incorrecto, una categoría desconocida y una modificación de esquema. El batch original no se sobrescribe y solamente se guarda el reporte del incidente.

In [44]:
!"{sys.executable}" -m src.quality_simulation --reference resultado_pipeline/adult_clean.csv --batch resultado_pipeline/monitoring/production_batch_1.csv --output resultado_pipeline/monitoring/quality_incident.json

{
  "simulation_only": true,
  "original_batch_permanently_modified": false,
  "original_fingerprint_unchanged": true,
  "contaminated_rows": 1001,
  "checks": {
    "missing_values": {
      "detected": true,
      "count": 153
    },
    "duplicated_rows": {
      "detected": true,
      "count": 1
    },
    "extreme_outlier": {
      "detected": true,
      "details": {
        "capital-gain": 1
      }
    },
    "incorrect_datatype": {
      "detected": true,
      "details": {
        "age": 1
      }
    },
    "unknown_category": {
      "detected": true,
      "details": {
        "native-country": [
          "UNKNOWN_NEW_COUNTRY"
        ]
      }
    },
    "schema_modification": {
      "detected": true,
      "missing_columns": [],
      "extra_columns": [
        "unexpected_schema_column"
      ]
    }
  },
  "detected_incidents": [
    "missing_values",
    "duplicated_rows",
    "extreme_outlier",
    "incorrect_datatype",
    "unknown_category",
    "schema_modifica

### Q.1 Resultado: Detecta → Bloquea → Registra

La decisión esperada es `BLOCK`, los seis incidentes deben aparecer como detectados y `original_fingerprint_unchanged` debe ser `True`.

In [45]:
quality_path = PROJECT_DIR / 'resultado_pipeline' / 'monitoring' / 'quality_incident.json'
quality_report = json.loads(quality_path.read_text(encoding='utf-8'))
quality_table = pd.DataFrame([
    {'problema': name, 'detectado': result['detected'], 'detalle': result.get('details', result.get('count', ''))}
    for name, result in quality_report['checks'].items()
])
display(quality_table)
print('Decisión del pipeline:', quality_report['pipeline_decision'])
print('Incidente registrado:', quality_report['incident_registered'])
print('Batch original intacto:', quality_report['original_fingerprint_unchanged'])
print('Batch contaminado guardado:', quality_report['contaminated_batch_saved'])

,problema,detectado,detalle
0,missing_values,True,153
1,duplicated_rows,True,1
2,extreme_outlier,True,{'capital-gain': 1}
3,incorrect_datatype,True,{'age': 1}
4,unknown_category,True,{'native-country': ['UNKNOWN_NEW_COUNTRY']}
5,schema_modification,True,


Decisión del pipeline: BLOCK
Incidente registrado: True
Batch original intacto: True
Batch contaminado guardado: False


## R. Estrategia de reentrenamiento

El PDF de la materia exige justificar cuándo se recomienda reentrenar, sin activarlo automáticamente solo por drift (Data Drift ≠ Model Degradation). `src/retrain_trigger.py` implementa esta regla: por cada lote evalúa **drift** (PSI máximo > 0.25, el mismo corte que usa `monitoring._psi_level` para "drift_alto"), **degradación de desempeño** (F1 con ground truth < 0.60) y **volumen mínimo** (≥ 500 filas). Solo si las tres condiciones se cumplen a la vez recomienda `RETRAIN_RECOMMENDED`, y siempre marca `requires_manual_approval: true` porque el reentrenamiento final requiere aprobación operativa, no se dispara solo.

In [46]:
!"{sys.executable}" -m src.retrain_trigger --batches-dir resultado_pipeline/monitoring --output resultado_pipeline/monitoring/retrain_decision.json

[
  {
    "batch_id": 1,
    "drift_strength": 0.0,
    "psi_max": 0.0861,
    "psi_max_feature": "native-country",
    "drift_condition_met": false,
    "performance_metric": "f1",
    "performance_value": 0.7928994082840237,
    "performance_condition_met": false,
    "production_rows": 1000,
    "volume_condition_met": true,
    "recommendation": "NO_RETRAIN_NEEDED",
    "requires_manual_approval": true
  },
  {
    "batch_id": 2,
    "drift_strength": 0.2,
    "psi_max": 0.1252,
    "psi_max_feature": "hours-per-week",
    "drift_condition_met": false,
    "performance_metric": "f1",
    "performance_value": 0.6452702702702703,
    "performance_condition_met": false,
    "production_rows": 1000,
    "volume_condition_met": true,
    "recommendation": "NO_RETRAIN_NEEDED",
    "requires_manual_approval": true
  },
  {
    "batch_id": 3,
    "drift_strength": 0.4,
    "psi_max": 0.4461,
    "psi_max_feature": "hours-per-week",
    "drift_condition_met": true,
    "performance_metric":

### R.1 Resultado de la decisión por lote

Se espera observar que los lotes con `drift_strength` bajo (1 y 2) queden en `NO_RETRAIN_NEEDED`, y que a partir del lote donde el PSI supera 0.25 y el F1 cae por debajo de 0.60 la recomendación cambie a `RETRAIN_RECOMMENDED`.

In [47]:
decision_path = PROJECT_DIR / 'resultado_pipeline' / 'monitoring' / 'retrain_decision.json'
decision = pd.read_json(decision_path)
display(decision[['batch_id', 'drift_strength', 'psi_max', 'psi_max_feature', 'performance_value', 'recommendation', 'requires_manual_approval']])

,batch_id,drift_strength,psi_max,psi_max_feature,performance_value,recommendation,requires_manual_approval
0,1,0.0,0.0861,native-country,0.792899,NO_RETRAIN_NEEDED,True
1,2,0.2,0.1252,hours-per-week,0.645270,NO_RETRAIN_NEEDED,True
2,3,0.4,0.4461,hours-per-week,0.590563,RETRAIN_RECOMMENDED,True
3,4,0.6,0.8292,hours-per-week,0.530055,RETRAIN_RECOMMENDED,True
4,5,0.8,1.3991,hours-per-week,0.437824,RETRAIN_RECOMMENDED,True
5,6,1.0,2.7408,age,0.418242,RETRAIN_RECOMMENDED,True


## Equivalente desde PowerShell

Si desea iniciar la API fuera del notebook, use una terminal independiente:

```powershell
./.venv/Scripts/Activate.ps1
python -m uvicorn src.api.main:app --reload --host 127.0.0.1 --port 8001
```

En otra terminal puede consultar las métricas:

```powershell
Invoke-RestMethod -Uri "http://127.0.0.1:8001/monitoring/system" -Method Get
```